In [1]:
import os

In [2]:
%pwd

'c:\\Users\\2021ICTS11\\Documents\\Krish Naik Projects\\first_end_to_end-DS_project\\research'

In [3]:
os.chdir("../")
%pwd

'c:\\Users\\2021ICTS11\\Documents\\Krish Naik Projects\\first_end_to_end-DS_project'

In [4]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [5]:
from src.datascience.constants import *
from src.datascience.utils.common import read_yaml, create_directories

In [6]:
class ConfigurationManager:
    def __init__(self, 
                config_filepath=CONFIG_FILE_PATH,
                params_filepath=PARAMS_FILE_PATH,
                schema_filepath=SCHEMA_FILE_PATH):
        
        self.config = read_yaml(config_filepath) 
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=Path(config.root_dir),
            source_URL=config.source_URL,
            local_data_file=Path(config.local_data_file),
            unzip_dir=Path(config.unzip_dir)
        )

        return data_ingestion_config

In [7]:
import os
import urllib.request as request
from src.datascience import logger
import zipfile

In [8]:
##component data ingestion

class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config


##download file file

    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url=self.config.source_URL,
                filename=self.config.local_data_file
            )
            print(f"{filename} downloaded! with following info: \n{headers}")
        else:
            print(f"File already exists of size: {round(os.path.getsize(self.config.local_data_file)/1024**2, 2)} MB")

    
    def extract_zip_file(self):
        import zipfile
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)

In [9]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()

except Exception as e:
    raise e



[2026-07-22 09:33:06,790: INFO: common]: yaml file: config\config.yaml loaded successfully]
[2026-07-22 09:33:06,791: INFO: common]: yaml file: params.yaml loaded successfully]
[2026-07-22 09:33:06,793: INFO: common]: yaml file: schema.yaml loaded successfully]
[2026-07-22 09:33:06,794: INFO: common]: Directory created at: artifacts]
[2026-07-22 09:33:06,795: INFO: common]: Directory created at: artifacts/data_ingestion]
artifacts\data_ingestion\data.zip downloaded! with following info: 
Connection: close
Content-Length: 23329
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "c69888a4ae59bc5a893392785a938ccd4937981c06ba8a9d6a21aa52b4ab5b6e"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny
X-XSS-Protection: 1; mode=block
X-GitHub-Request-Id: 9458:3E8144:35211:73036:6A60412E
Accept-Ranges: bytes
Date: Wed, 22 Jul 2026 04:03:59 GMT
Via: 1.1 varnish